# ML-08 — Capstone Modeling Lane

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/abdulwasay45/flyrankinternship/blob/main/work/notebooks/w05_model.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Method choice and why

*Which method from the toolkit, and why it fits your lane.*

**Lane:** Refresh / Content Opportunity Scoring  
**Question shape:** “Which pages first?” → ranking / scoring problem.

**Methods I will train (in order of complexity):**
1. **Logistic Regression** — readable coefficients, strong baseline for binary labels.
2. **Decision Tree (max_depth=5)** — fully readable if/else rules.
3. **Random Forest** — stronger ensemble that still gives feature importances.

**Why these three**  
They match the menu from the live session and the starter pipeline. I start simple (LR + shallow tree) and only keep Random Forest if it clearly beats the transparent baseline on Precision@50 under client-holdout. Complexity is not rewarded by itself.

**Target:** `is_declining_label` (observed: trend_direction == "down")  
**Primary metric:** Precision@50 (matches the real decision capacity of an editor)  
**Secondary:** Precision@20, ROC-AUC, average precision, base rate.

In [3]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 2. Split design

*Grouped by client? Time-aware? Say why this split is honest for your question.*

**Split:** client-holdout (GroupShuffle-style)

~20% of unique clients are held out entirely for test.  
No page from a test client ever appears in training.  
This is the same design the reference pipeline uses and matches the real deployment question (“will this work on a new client?”).

If client count is too small for a clean holdout, fall back to stratified row holdout and state it clearly.

In [4]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.


## 3. Train + compare vs my baseline

*Same data, same metric, same split as your Week-4 baseline. Show the table.*

In [5]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
import pandas as pd
import numpy as np
import os, sys, subprocess, json
from pathlib import Path

from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.ensemble import RandomForestClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.model_selection import train_test_split
from sklearn.metrics import roc_auc_score, average_precision_score

# ---------- path setup ----------
IN_COLAB = "google.colab" in sys.modules
if IN_COLAB:
    REPO_URL = "https://github.com/abdulwasay45/flyrankinternship.git"
    REPO_DIR = "flyrankinternship"
    if not os.path.isdir(REPO_DIR):
        subprocess.run(["git", "clone", "--depth", "1", REPO_URL, REPO_DIR], check=True)
    os.chdir(REPO_DIR)
else:
    while not os.path.exists("data/raw/content_refresh_anonymized.csv") and os.getcwd() != "/":
        os.chdir("..")

df = pd.read_csv("data/raw/content_refresh_anonymized.csv")
df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
print(f"Loaded {len(df):,} pages | declining rate: {df['is_declining_label'].mean():.1%}")

# ---------- features (safe, no label leakage) ----------
numeric_features = [
    "search_volume", "competition", "cpc", "word_count", "char_count",
    "impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d",
    "days_with_impressions", "days_with_sessions",
    "content_age_days", "days_since_last_update",
    "ctr", "avg_position", "engagement_rate", "scroll_rate", "ai_traffic_pct",
]
categorical_features = [
    "competition_level", "content_type", "main_intent",
    "age_tier", "freshness_tier", "word_count_tier",
    "impression_tier", "position_tier",
]

# keep only columns that exist
numeric_features = [c for c in numeric_features if c in df.columns]
categorical_features = [c for c in categorical_features if c in df.columns]

X_num = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
# log1p for heavy-tailed volume columns
for col in ["impressions_90d", "clicks_90d", "sessions_90d", "ai_sessions_90d"]:
    if col in X_num.columns:
        X_num[f"log_{col}"] = np.log1p(X_num[col])

X_cat = pd.get_dummies(df[categorical_features].fillna("unknown").astype(str), dtype=float)
X = pd.concat([X_num.reset_index(drop=True), X_cat.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)
feature_columns = list(X.columns)

# ---------- client-holdout split ----------
RANDOM_STATE = 42
client_series = df["client_id"].fillna("unknown").astype(str)
unique_clients = client_series.drop_duplicates().to_numpy()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(unique_clients)
n_test_clients = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test_clients])
test_mask = client_series.isin(test_clients).to_numpy()
train_idx = np.where(~test_mask)[0]
test_idx  = np.where(test_mask)[0]
split_strategy = "client_holdout"

# safety fallback
if len(train_idx) == 0 or len(test_idx) == 0 or y.iloc[train_idx].nunique() < 2 or y.iloc[test_idx].nunique() < 2:
    train_idx, test_idx = train_test_split(np.arange(len(df)), test_size=0.2, random_state=RANDOM_STATE, stratify=y)
    split_strategy = "stratified_row_holdout"

print(f"Split: {split_strategy} | train={len(train_idx):,} | test={len(test_idx):,}")
print(f"Test clients: {n_test_clients if split_strategy=='client_holdout' else 'n/a'}")

X_tr, X_te = X.iloc[train_idx], X.iloc[test_idx]
y_tr, y_te = y.iloc[train_idx], y.iloc[test_idx]

# ---------- rebuild the SAME baseline score on the test set ----------
def baseline_score(frame):
    stale   = (frame["days_since_last_update"] >= 180).astype(int)
    visible = (frame["impressions_90d"] >= 500).astype(int)
    low_ctr = (
        (frame["impressions_90d"] >= 500) &
        (frame["avg_position"] > 0) & (frame["avg_position"] <= 20) &
        (frame["ctr"] < 0.5)
    ).astype(int)
    score = stale * visible * np.log1p(frame["impressions_90d"]) + low_ctr * np.log1p(frame["impressions_90d"]) * 0.5
    return score

base_te = baseline_score(df.iloc[test_idx]).to_numpy()

def precision_at_k(y_true, scores, k):
    order = np.argsort(-np.asarray(scores))
    topk = np.asarray(y_true)[order[:k]]
    return float(topk.mean()) if len(topk) else 0.0

def metrics(y_true, scores, prefix=""):
    return {
        f"{prefix}precision_at_20": precision_at_k(y_true, scores, 20),
        f"{prefix}precision_at_50": precision_at_k(y_true, scores, 50),
        f"{prefix}precision_at_100": precision_at_k(y_true, scores, 100),
        f"{prefix}roc_auc": float(roc_auc_score(y_true, scores)) if y_true.nunique() == 2 else 0.0,
        f"{prefix}avg_precision": float(average_precision_score(y_true, scores)) if y_true.nunique() == 2 else 0.0,
    }

base_metrics = metrics(y_te, base_te, prefix="baseline_")
print("\nBaseline on test set:")
for k, v in base_metrics.items():
    print(f"  {k}: {v:.3f}")
print(f"  base rate (test): {y_te.mean():.3f}")

# ---------- train the three models ----------
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE)),
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results = {}
probs = {}
for name, model in models.items():
    model.fit(X_tr, y_tr)
    p = model.predict_proba(X_te)[:, 1]
    probs[name] = p
    results[name] = metrics(y_te, p)
    print(f"\n{name}:")
    for k, v in results[name].items():
        print(f"  {k}: {v:.3f}")

# ---------- comparison table ----------
print("\n" + "="*60)
print("MODEL vs BASELINE (same client-holdout test set)")
print("="*60)
rows = []
rows.append({
    "model": "baseline_rule",
    "precision_at_20": base_metrics["baseline_precision_at_20"],
    "precision_at_50": base_metrics["baseline_precision_at_50"],
    "precision_at_100": base_metrics["baseline_precision_at_100"],
    "roc_auc": base_metrics["baseline_roc_auc"],
    "avg_precision": base_metrics["baseline_avg_precision"],
})
for name, m in results.items():
    rows.append({
        "model": name,
        "precision_at_20": m["precision_at_20"],
        "precision_at_50": m["precision_at_50"],
        "precision_at_100": m["precision_at_100"],
        "roc_auc": m["roc_auc"],
        "avg_precision": m["avg_precision"],
    })
table = pd.DataFrame(rows).sort_values("precision_at_50", ascending=False)
print(table.round(3).to_string(index=False))
print(f"\nTest base rate: {y_te.mean():.3f}")
print(f"Split strategy: {split_strategy}")

# pick best by Precision@50
best_name = table.iloc[0]["model"]
print(f"Best by Precision@50: {best_name}")

# save a small metrics receipt
os.makedirs("work/outputs", exist_ok=True)
receipt = {
    "split_strategy": split_strategy,
    "test_base_rate": float(y_te.mean()),
    "baseline": base_metrics,
    "models": results,
    "best_model": best_name,
}
with open("work/outputs/w05_model_metrics.json", "w") as f:
    json.dump(receipt, f, indent=2)
print("\nWrote work/outputs/w05_model_metrics.json")

Loaded 30,000 pages | declining rate: 54.2%
Split: client_holdout | train=27,675 | test=2,325
Test clients: 6

Baseline on test set:
  baseline_precision_at_20: 0.200
  baseline_precision_at_50: 0.300
  baseline_precision_at_100: 0.350
  baseline_roc_auc: 0.557
  baseline_avg_precision: 0.410
  base rate (test): 0.391

logistic_regression:
  precision_at_20: 0.350
  precision_at_50: 0.320
  precision_at_100: 0.410
  roc_auc: 0.700
  avg_precision: 0.519

decision_tree:
  precision_at_20: 0.500
  precision_at_50: 0.560
  precision_at_100: 0.580
  roc_auc: 0.742
  avg_precision: 0.575

random_forest:
  precision_at_20: 0.800
  precision_at_50: 0.720
  precision_at_100: 0.770
  roc_auc: 0.751
  avg_precision: 0.624

MODEL vs BASELINE (same client-holdout test set)
              model  precision_at_20  precision_at_50  precision_at_100  roc_auc  avg_precision
      random_forest             0.80             0.72              0.77    0.751          0.624
      decision_tree             0.50

## 4. Errors and interpretation

*Where is the model wrong? What does it lean on? A short error analysis beats a big metric table.*

**What the comparison shows**  
The model table above is computed on the exact same client-holdout test pages as the baseline.  
If Random Forest (or Logistic Regression) wins on Precision@50, the lift is real under the deployment-style split.  
If the baseline stays competitive at Precision@20, that is also a finding — a sharp human rule can still be excellent at the very top of the list.

**Feature interpretation**  
Top signals that usually matter for this lane (from the starter pipeline and from this run):
- days_with_impressions / log_impressions — volume is the strongest prior
- days_since_last_update / content_age_days — staleness
- avg_position / position_tier — visibility context
- ctr — engagement quality relative to position

**Where the model is most wrong**  
- High-volume evergreen pages that are intentionally left static (false positives for “refresh”).  
- Low-volume pages where CTR and trend are noisy (false negatives or unstable ranks).  
- Clients whose traffic patterns differ sharply from the training clients (the whole point of client-holdout).

**Three concrete error patterns to watch**  
1. Page ranks high because of age + volume, but trend is actually flat → editor may waste a slot.  
2. Page has weak CTR for its position but is already optimized → “refresh_and_review_ctr” is wrong.  
3. Page is declining for competitive reasons the features cannot see → model cannot invent missing context.

All claims stay directional and decision-support only. No causal language.

In [6]:
# This cell is for CODE (numbers, a query, a check).
# Write your text answer in the cell ABOVE this one — typing sentences here breaks Run All.
# Feature importance from the best model (if it is RF or tree)
best_model_obj = models.get(best_name) if best_name in models else models["random_forest"]
if hasattr(best_model_obj, "feature_importances_"):
    imp = pd.Series(best_model_obj.feature_importances_, index=feature_columns)
elif hasattr(best_model_obj, "named_steps"):
    coef = np.abs(best_model_obj.named_steps["model"].coef_[0])
    imp = pd.Series(coef, index=feature_columns)
else:
    imp = pd.Series(dtype=float)

print("Top 10 features:")
print(imp.sort_values(ascending=False).head(10).round(4))

Top 10 features:
days_with_impressions    0.1122
impressions_90d          0.1006
avg_position             0.0994
log_impressions_90d      0.0973
content_age_days         0.0814
word_count               0.0364
char_count               0.0363
age_tier_365+            0.0347
scroll_rate              0.0319
log_clicks_90d           0.0313
dtype: float64


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.